# NeuroLens: Complete Three-Condition Research Training Pipeline

**Research question:** can a shared representation space connect a 2D brain MRI slice to the 3D MRI volume from the same patient while supporting tumour classification and segmentation?

This notebook is built around the repository implementation. BraTS supplies genuine same-patient 2D/3D pairs and voxel labels. BRISC supplies independent 2D T1 classification and segmentation records; it is audited and inspected separately and never treated as paired with BraTS. The checked-in architecture, losses, dataset preprocessing, and saved patient splits remain the source of truth.

The known integration evidence is documented rather than rerun here: 371 complete BraTS cases (296/37/38 saved patients), four MRI modalities, 64-cubed real batch tensors, finite loss, and 108/108 trainable tensors receiving finite gradients in the earlier gradient check. BRISC's earlier audit found 6,000 classification images and 4,793 valid segmentation pairs with zero classification/segmentation image-content overlap. Runtime cells below verify current Drive availability and report current data-derived values.

## 1. NeuroLens research objective

### Research workflow

```mermaid
flowchart TD
  A[Mount Drive and inspect repository] --> B[Read-only BRISC and BraTS audits]
  B --> C[Verify saved patient-level manifests]
  C --> D[Normalize and crop BraTS modalities; keep labels aligned]
  D --> E[Same-patient 2D slice + 3D volume DataLoader]
  E --> F[Existing NeuroLens 2D and 3D encoders]
  F --> G[Binary tumour classification + 4-class segmentation]
  F --> H[Shared projected 2D/3D embeddings]
  G --> I[Condition A multitask optimization]
  I --> J[Validation selects best checkpoint]
  J --> K[Held-out test evaluation and research report]
  R[Independent 2D BRISC tasks] -. no cross-dimensional pairing .-> B
```

**Methodology:** a 2D encoder summarizes a slice and a 3D encoder summarizes the corresponding patient crop. Projection heads map both features into the common embedding dimension. The 3D decoder produces voxel logits. Condition A is the controlled baseline; according to the trainer it optimizes classification and segmentation terms, with contrastive and consistency weights disabled. Alignment metrics may still be reported on genuine paired BraTS embeddings, but they are not Condition A training objectives.

## 2. Experimental design: Conditions A, B, and C

Each condition is a fresh experiment with a fresh NeuroLens instance and AdamW optimizer. All three use the exact saved BraTS patient manifests, same preprocessing, same seed, batch size 2, 50 epochs, optimizer settings, deterministic validation/test slice, and evaluation code. The model architecture does not change.

| Condition | Classification | Segmentation | Symmetric InfoNCE | Cosine consistency |
|---|---:|---:|---:|---:|
| A — multitask baseline | 1.0 | 1.0 | 0.0 | 0.0 |
| B — contrastive alignment | 1.0 | 1.0 | 0.1 | 0.0 |
| C — full NeuroLens | 1.0 | 1.0 | 0.1 | 0.05 |

Only genuine same-patient BraTS slice/volume pairs are positives. In B/C, other patients in the same batch are InfoNCE negatives; aligned train loaders drop incomplete singleton batches. BRISC is not used in these experiments. The master cell runs A, B, then C and evaluates the held-out test set only after all three best checkpoints have been selected on validation.

## 3. Colab environment setup

Mount Drive, locate the repository in a small set of known locations (or clone it only when absent), check the existing Python/PyTorch/CUDA stack, install only `requirements.txt`, and seed all supported RNGs. GPU use is recommended for the 3D path; CPU remains usable for setup and read-only inspection.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, platform, random

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Colab; Drive mounting is unavailable in this runtime.")

DRIVE_PROJECT = Path("/content/drive/MyDrive/NeuroLens")
REPO_CANDIDATES = [Path("/content/Neurolens"), DRIVE_PROJECT, Path.cwd()]
REPO = next((p for p in REPO_CANDIDATES if (p / "config.yaml").is_file() and (p / "neurolens").is_dir()), None)
if REPO is None:
    REPO = Path("/content/Neurolens")
    if not Path("/content").exists():
        raise RuntimeError("Expected Colab /content directory or an existing NeuroLens checkout.")
    subprocess.run(["git", "clone", "https://github.com/MSSG-07/Neurolens.git", str(REPO)], check=True)
os.chdir(REPO)
print("Repository:", REPO, "| Python:", sys.version.split()[0], "| Platform:", platform.platform())

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is not installed in this runtime. Select a standard Colab runtime; this notebook does not force-install or replace PyTorch.")
import torch
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print("WARNING: no CUDA GPU detected. Inspection is fine; the training cell requires CUDA by default.")

requirements = REPO / "requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Project requirements file is missing: {requirements}")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
from neurolens.utils.reproducibility import seed_everything
SEED = 42
seed_everything(SEED)
print("Seed configured:", SEED, "| deterministic cuDNN:", torch.backends.cudnn.deterministic)


## 4. Repository and implementation inspection

The source is inspected from this checkout, not paraphrased from a generic model template. The output lists the package files and prints the configuration and principal entry points so configuration drift is visible before data access.

In [ ]:
from pathlib import Path

for path in sorted((REPO / "neurolens").rglob("*.py")):
    print(path.relative_to(REPO))
for relative in ["config.yaml", "train.py", "evaluate.py", "neurolens/training/train.py",
                 "neurolens/datasets/brats.py", "neurolens/datasets/transforms.py",
                 "neurolens/models/encoder2d.py", "neurolens/models/encoder3d.py",
                 "neurolens/models/alignment.py", "neurolens/models/multitask.py",
                 "neurolens/losses/contrastive.py", "neurolens/losses/multitask_loss.py",
                 "neurolens/training/trainer.py", "neurolens/training/checkpoint.py",
                 "neurolens/evaluation/classification.py", "neurolens/evaluation/segmentation.py",
                 "neurolens/evaluation/alignment.py", "neurolens/utils/metrics.py"]:
    print(f"\n{'='*18} {relative} {'='*18}\n")
    print((REPO / relative).read_text(encoding="utf-8"))


## 5. Dataset organization and read-only auditing

The paths below are the known Drive locations. The notebook checks rather than creates dataset files. BRISC resolution uses the repository `BRISCPathResolver`, which checks the manifest relative path and the known canonical/`(1)` directory pair. The complete content-hash audit, including overlap and image-mask checks, was already completed in prior integration work; this notebook avoids repeating that full scan and performs only path/manifest and representative loader checks.

In [ ]:
from pathlib import Path
import csv, json, collections

BRATS_ROOT = DRIVE_PROJECT / "data/raw/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
BRISC_ROOT = DRIVE_PROJECT / "data/raw/BRISC2025/brisc2025"
MANIFEST_DIR = DRIVE_PROJECT / "manifests"
TRAIN_MANIFEST = MANIFEST_DIR / "train.csv"
VAL_MANIFEST = MANIFEST_DIR / "validation.csv"
TEST_MANIFEST = MANIFEST_DIR / "test.csv"
BRISC_MANIFEST = BRISC_ROOT / "manifest.csv"

required_paths = {"BraTS root": BRATS_ROOT, "BRISC root": BRISC_ROOT,
                  "BraTS train manifest": TRAIN_MANIFEST, "BraTS validation manifest": VAL_MANIFEST,
                  "BraTS test manifest": TEST_MANIFEST, "BRISC manifest": BRISC_MANIFEST}
for label, path in required_paths.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'} — {path}")
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Required Drive inputs are unavailable. Mount the expected Drive and verify these paths:\n" + "\n".join(missing))

print("\nBRISC task-directory variants:")
for name in ("classification_task", "classification_task (1)", "segmentation_task", "segmentation_task (1)"):
    path = BRISC_ROOT / name
    print(name, "exists=", path.is_dir(), "top-level entries=", len(list(path.iterdir())) if path.is_dir() else 0)

with BRISC_MANIFEST.open(newline="", encoding="utf-8-sig") as f:
    brisc_rows = list(csv.DictReader(f))
if not brisc_rows:
    raise ValueError("BRISC manifest is empty")
print("\nBRISC manifest rows:", len(brisc_rows), "columns:", sorted(brisc_rows[0]))
split_key = next((k for k in ("split", "subset", "partition") if k in brisc_rows[0]), None)
if split_key is None:
    raise ValueError("BRISC manifest has no explicit split column; refusing to invent a split")
print("Split counts:", dict(collections.Counter(r[split_key].strip().casefold() for r in brisc_rows)))
print("Tumor-label counts (all manifest records):", dict(collections.Counter(r.get("tumor_label", "").strip().casefold() for r in brisc_rows)))
print("Mask flags:", dict(collections.Counter(r.get("is_mask", "") for r in brisc_rows)))
brisc_json = BRISC_ROOT / "manifest.json"
if brisc_json.is_file():
    manifest_json_data = json.loads(brisc_json.read_text(encoding="utf-8"))
    print("manifest.json top-level type/keys:", type(manifest_json_data).__name__, list(manifest_json_data)[:20] if isinstance(manifest_json_data, dict) else len(manifest_json_data))
else:
    print("manifest.json is absent; CSV remains the dataset loader's manifest input.")
import hashlib
for manifest_file in (BRISC_MANIFEST, brisc_json):
    checksum_candidates = (manifest_file.with_suffix(manifest_file.suffix + ".sha256"), manifest_file.with_suffix(".sha256"))
    checksum_file = next((p for p in checksum_candidates if p.is_file()), None)
    if checksum_file:
        expected_hash = checksum_file.read_text(encoding="utf-8").split()[0]
        actual_hash = hashlib.sha256(manifest_file.read_bytes()).hexdigest()
        checksum_matches = expected_hash.casefold() == actual_hash.casefold()
        print(manifest_file.name, "checksum matches:", checksum_matches)
        if not checksum_matches: raise ValueError(f"BRISC manifest checksum mismatch for {manifest_file}")
    else:
        print(manifest_file.name, "checksum file: not present")
print("BRISC audit facts from prior completed integration: 6,000 classification images; 4,793 segmentation images/masks; 4,793 valid pairs; 15,586 manifest rows; zero classification/segmentation image-content overlap.")

from neurolens.datasets.brisc import BRISCDataset, BRISCPathResolver
brisc_resolver = BRISCPathResolver(BRISC_ROOT)
example_relative = next((r.get("relative_path", "") for r in brisc_rows if r.get("relative_path")), None)
if example_relative:
    print("Resolver example:", example_relative, "->", brisc_resolver.resolve(example_relative))


## 6. BRISC independent dataset audit

`BRISCDataset(root, manifest, task, split)` reads official manifest records and the split explicitly. Classification labels use the loader’s named mapping (`glioma`, `meningioma`, `pituitary`, `no_tumor`); segmentation masks are returned as masks. Prior loader checks produced one-channel 512×512 images and 512×512 masks. Since the checked-in NeuroLens model expects four input channels and a binary class head, BRISC is not fed to that model in this baseline. Adapting it would require a separate, explicitly designed experiment and must not create any BraTS correspondence.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from neurolens.datasets.brisc import BRISCPathResolver

# Read-only, representative check only: the full 15,586-row content/pair audit
# and BRISC DataLoader tests were completed previously and are not repeated.
resolver = BRISCPathResolver(BRISC_ROOT)
def manifest_bool(value):
    text = str(value).strip().casefold()
    if text in {"1", "true", "yes", "y"}: return True
    if text in {"0", "false", "no", "n"}: return False
    raise ValueError(f"Invalid BRISC is_mask value: {value!r}")
def normalized_manifest_path(row):
    value = str(row.get("relative_path", "") or "").strip()
    if not value: raise ValueError("BRISC representative row is missing relative_path")
    return value.replace("\\", "/")

images = [r for r in brisc_rows if not manifest_bool(r["is_mask"])]
masks = [r for r in brisc_rows if manifest_bool(r["is_mask"])]
classification_images = [r for r in images if path_task(normalized_manifest_path(r)) in {"classification_task", "classification_task (1)"}]
segmentation_images = [r for r in images if path_task(normalized_manifest_path(r)) in {"segmentation_task", "segmentation_task (1)"}]
segmentation_masks = [r for r in masks if path_task(normalized_manifest_path(r)) in {"segmentation_task", "segmentation_task (1)"}]
actual_counts = (len(brisc_rows), len(classification_images), len(segmentation_images), len(segmentation_masks))
expected_counts = (15586, 6000, 4793, 4793)
print("BRISC manifest count check (rows, classification images, segmentation images, segmentation masks):", actual_counts)
if actual_counts != expected_counts: raise RuntimeError(f"BRISC manifest counts differ from established audit: expected {expected_counts}, got {actual_counts}")
print("BRISC classification label counts:", dict(collections.Counter(str(r["tumor_label"]).strip().casefold() for r in classification_images)))
print("BRISC segmentation labels in manifest:", sorted({str(r["tumor_label"]).strip().casefold() for r in segmentation_images}))
def path_task(relative_path): return relative_path.split("/", 1)[0]
class_row = next(r for r in images if path_task(normalized_manifest_path(r)) in {"classification_task", "classification_task (1)"})
mask_row = next(r for r in masks if path_task(normalized_manifest_path(r)) in {"segmentation_task", "segmentation_task (1)"})
mask_split = str(mask_row[split_key]).strip().casefold()
linked_value = str(mask_row.get("linked_image", "") or "").replace("\\", "/")
linked_basename = linked_value.rsplit("/", 1)[-1].casefold()
linked_rows = [r for r in images
    if str(r[split_key]).strip().casefold() == mask_split
    and path_task(normalized_manifest_path(r)) in {"segmentation_task", "segmentation_task (1)"}
    and normalized_manifest_path(r).rsplit("/", 1)[-1].casefold() == linked_basename]
if len(linked_rows) != 1:
    raise RuntimeError(f"Expected one same-split segmentation image manifest row for linked_image={linked_value!r}; found {len(linked_rows)}")
class_path = resolver.resolve(normalized_manifest_path(class_row))
mask_path = resolver.resolve(normalized_manifest_path(mask_row))
linked_path = resolver.resolve(normalized_manifest_path(linked_rows[0]))
print("Representative classification row:", {k: class_row.get(k) for k in ("relative_path", "filename", "task", split_key, "tumor_label", "is_mask")})
print("Resolved classification image:", class_path)
print("Representative segmentation mask row:", {k: mask_row.get(k) for k in ("relative_path", "filename", "task", split_key, "linked_image", "is_mask")})
print("Resolved mask:", mask_path, "| same-split linked image:", linked_path)
with Image.open(class_path) as image_file:
    class_image = np.asarray(image_file.convert("L"))
with Image.open(linked_path) as image_file:
    seg_image = np.asarray(image_file.convert("L"))
with Image.open(mask_path) as mask_file:
    seg_mask = np.asarray(mask_file)
if seg_image.shape[:2] != seg_mask.shape[:2]:
    raise ValueError(f"Representative BRISC image/mask dimensions differ: {seg_image.shape} vs {seg_mask.shape}")
print("Readable representative shapes:", {"classification": class_image.shape, "segmentation_image": seg_image.shape,
      "mask": seg_mask.shape, "mask_values": np.unique(seg_mask).tolist()})
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(class_image, cmap="gray"); axes[0].set_title(f"BRISC classification: {class_row['tumor_label']}")
axes[1].imshow(seg_image, cmap="gray"); axes[1].set_title("BRISC segmentation image")
axes[2].imshow(seg_mask, cmap="viridis"); axes[2].set_title("Manifest-linked mask")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("Previous BRISC DataLoader validation: 6,000 classification images and 4,793 valid segmentation pairs; no BRISC samples enter BraTS training.")


## 7. BraTS saved split verification

The saved patient lists are loaded by `read_patient_ids`; `validate_saved_splits` rejects intersections. These files are not regenerated or edited. Patient-level separation keeps slices, patches, and correlated anatomy from one patient out of other partitions. Validation controls checkpoint selection; test stays unopened until the final evaluation section.

In [ ]:
from neurolens.config import load_config
from neurolens.training.train import validate_saved_splits, make_datasets
from neurolens.datasets.manifests import read_patient_ids

cfg = load_config(str(REPO / "config.yaml"))
cfg.brats_root = str(BRATS_ROOT)
cfg.data_root = str(BRATS_ROOT)
cfg.manifest_train, cfg.manifest_val, cfg.manifest_test = map(str, (TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST))
cfg.modalities = ("t1n", "t1c", "t2w", "t2f")
cfg.patch_size = (64, 64, 64)
cfg.slice_axis = 2
cfg.batch_size = 2
cfg.num_workers = 2
cfg.accumulation_steps = 4
cfg.epochs = 50
cfg.learning_rate = 1e-4
cfg.weight_decay = 1e-5
cfg.projection_dim = 256
cfg.num_classes = 4
cfg.classification_classes = 2
cfg.base_channels = 8
cfg.temperature = 0.1
cfg.lambda_classification = 1.0
cfg.lambda_segmentation = 1.0
cfg.lambda_contrastive = 0.1
cfg.lambda_consistency = 0.05
cfg.output_dir = str(DRIVE_PROJECT / "experiments" / "condition_A")
cfg.alignment_condition = "A"
cfg.seed = 42
cfg.mixed_precision = True
cfg.device = "cuda"
cfg.validate()
split_counts = validate_saved_splits(cfg)
split_ids = {name: set(read_patient_ids(str(path), cfg.manifest_id_column)) for name, path in
             (("train", TRAIN_MANIFEST), ("validation", VAL_MANIFEST), ("test", TEST_MANIFEST))}
print("Saved split patient counts:", split_counts)
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = split_ids[left] & split_ids[right]
    print(f"{left}/{right} overlap:", len(overlap))
    assert not overlap, f"Patient leakage: {left}/{right} share {sorted(overlap)[:5]}"
assert split_counts == {"train": 296, "validation": 37, "test": 38}, f"Unexpected saved split counts: {split_counts}"
print("Split verification passed; the test IDs have not been used to create any training or validation data.")


## 8. MRI preprocessing and deterministic evaluation

`BraTSPairedDataset` loads each patient’s NIfTI modalities in `cfg.modalities` order (`t1n`, `t1c`, `t2w`, `t2f`) and the segmentation. It transposes NIfTI `(X,Y,Z)` arrays to `(Z,Y,X)`, applies per-channel nonzero-voxel z-score normalization, center-crops/pads images and labels to `patch_size`, and maps source label 4 to contiguous label 3. Training samples a random valid slice from the same cropped volume. Validation and test select the middle index of the sorted valid-slice list, deterministically; the label remains aligned with the 3D volume. This selection rule is shared across all conditions. No geometric augmentation is implemented in the current baseline. Do not apply ordinary intensity normalization or interpolating transforms to label masks.

In [ ]:
from neurolens.training.train import make_datasets

# Build saved train/validation datasets once; reuse them for the whole A/B/C study.
train_dataset, val_dataset = make_datasets(cfg)
print("Saved-patient datasets:", len(train_dataset), "train;", len(val_dataset), "validation")
sample = train_dataset[0]
print("Patient:", sample["patient_id"], "2D:", tuple(sample["image2d"].shape), "3D:", tuple(sample["image3d"].shape),
      "seg:", tuple(sample["segmentation"].shape), "labels:", torch.unique(sample["segmentation"]).tolist(),
      "slice index:", sample["slice_index"], "class label:", int(sample["class_label"]))
assert sample["image2d"].shape[0] == len(cfg.modalities)
assert tuple(sample["image3d"].shape[1:]) == tuple(cfg.patch_size)
assert sample["segmentation"].shape == sample["image3d"].shape[1:]
assert set(torch.unique(sample["segmentation"]).tolist()).issubset({0, 1, 2, 3})

# Paired image and label views use the exact same slice index and axis.
volume_np = sample["image3d"].numpy()
label_np = sample["segmentation"].numpy()
slice_img = sample["image2d"].numpy()
slice_mask = np.take(label_np, sample["slice_index"], axis=cfg.slice_axis)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(slice_img[0], cmap="gray"); axes[0].set_title("2D sampled t1n slice")
axes[1].imshow(slice_img[0], cmap="gray"); axes[1].imshow(np.ma.masked_where(slice_mask == 0, slice_mask), alpha=.45, vmin=1, vmax=3, cmap="viridis"); axes[1].set_title("Same-plane label overlay")
axes[2].imshow(volume_np[0, volume_np.shape[1]//2], cmap="gray"); axes[2].set_title("3D volume middle depth (t1n)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## 9. DataLoaders and tensor shapes

The trainer’s batches contain same-patient `image2d`, `image3d`, `segmentation`, and `class_label` values. Training shuffles patients; validation and test are ordered. No patient in validation or test can contribute a training slice. This cell constructs the train and validation loaders only; the test loader is deliberately deferred.

In [ ]:
import torch
from torch.utils.data import DataLoader

device = torch.device(cfg.device if cfg.device == "cpu" or torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                          pin_memory=pin, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                        pin_memory=pin, drop_last=False)
batch = next(iter(train_loader))
print("Device:", device, "| train/validation patients:", len(train_dataset), len(val_dataset))
assert train_dataset.random_slice is True
assert val_dataset.random_slice is False, "Validation must use deterministic slices"
print("Slice policy: random valid slice for training; middle valid slice for validation/test.")
for key, value in batch.items():
    if torch.is_tensor(value): print(f"{key}: shape={tuple(value.shape)} dtype={value.dtype} device={value.device}")
    elif isinstance(value, (list, tuple)): print(f"{key}: {value[:4]}")
print("Class labels in batch:", dict(zip(*np.unique(batch["class_label"].numpy(), return_counts=True))))
print("Segmentation labels in batch:", torch.unique(batch["segmentation"]).tolist())


## 10. Actual NeuroLens architecture

`NeuroLens` constructs `Encoder2D`, `Encoder3D`, and separate projection heads using the checked-in constructor. The 2D encoder is a sequence of strided convolution, depthwise convolution, pointwise convolution, batch normalization, and ReLU blocks followed by global pooling, a feature projection, and binary classifier. The 3D encoder uses four convolutional stages with pooling, a volume projection, and a decoder with transposed convolutions and skip connections to produce four segmentation logits per voxel. Projection heads map both pathway embeddings into the same configured dimension and L2-normalize them. The forward result keys are `classification_logits`, `embedding2d`, `embedding3d`, `segmentation_logits`, `features2d`, `pooled2d`, and `pooled3d`.

### Architecture diagram (implementation-faithful)

```mermaid
flowchart LR
  X2[Same-patient 2D slice\nB×4×H×W] --> E2[Encoder2D\nstrided/depthwise/pointwise CNN]
  E2 --> C[Binary classifier\nB×2 logits]
  E2 --> P2[2D projection head\nB×256 normalized]
  X3[Same-patient 3D crop\nB×4×D×H×W] --> E3[Encoder3D\n4 stages + pooling]
  E3 --> P3[3D projection head\nB×256 normalized]
  E3 --> U[3D decoder\nupsampling + skip connections]
  U --> S[4-class voxel logits\nB×4×D×H×W]
  P2 -. shared embedding space .- P3
  C --> L[MultiTaskLoss]
  S --> L
  P2 --> L
  P3 --> L
```

The diagonal pairing is the dataset invariant: row *i* in `image2d` and `image3d` belongs to the same BraTS patient. BRISC rows never enter these paired batches.

In [ ]:
from neurolens.models.multitask import NeuroLens
architecture_model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
print(architecture_model)
print("Trainable parameters:", sum(p.numel() for p in architecture_model.parameters() if p.requires_grad))
print("Configured output shapes for the inspected batch:", {"classification_logits": (cfg.batch_size, cfg.classification_classes), "embedding2d/embedding3d": (cfg.batch_size, cfg.projection_dim), "segmentation_logits": (cfg.batch_size, cfg.num_classes, *cfg.patch_size)})
print("The previously completed real-data forward/gradient checks are not repeated in this inspection cell.")


## 11. Loss functions and condition mapping

`MultiTaskLoss` uses cross-entropy for binary class logits, multiclass segmentation cross-entropy plus a soft Dice loss averaged over all four classes (including background), cosine embedding consistency, and symmetric in-batch InfoNCE. `Trainer` weights classification and segmentation from config; for Condition A it explicitly sets contrastive and consistency weights to zero. Thus the Condition A optimized objective is `1.0 × classification + 1.0 × segmentation`. The class target is tumour presence, not tumour subtype. Source segmentation labels are mapped to `0=background, 1=NCR, 2=ED, 3=ET`.

The optimizer in `Trainer` is AdamW with the config learning rate and weight decay. Gradient accumulation is applied over four steps; AMP uses a CUDA GradScaler when configured and available. The repository has no scheduler, gradient clipping, early stopping, or built-in best-checkpoint selection. A final small validation batch with one row can make InfoNCE undefined, but it is omitted for singleton batches; in Condition A its weight is zero anyway.

In [ ]:
from neurolens.losses.multitask_loss import MultiTaskLoss
CONDITION_WEIGHTS = {
    "A": {"classification": 1.0, "segmentation": 1.0, "contrastive": 0.0, "consistency": 0.0},
    "B": {"classification": 1.0, "segmentation": 1.0, "contrastive": 0.1, "consistency": 0.0},
    "C": {"classification": 1.0, "segmentation": 1.0, "contrastive": 0.1, "consistency": 0.05},
}
for condition, weights in CONDITION_WEIGHTS.items():
    loss_impl = MultiTaskLoss(weights, cfg.temperature)
    assert loss_impl.weights == weights
    print(condition, loss_impl.weights)
print("Segmentation CE + soft Dice averages over all four labels (including background).")
print("Classification metrics are binary tumour presence; not BRISC subtype labels.")
print("HD95 is a final per-case metric using the repository default unit voxel spacing; empty-region HD95 may be NaN.")


## 12. Training configuration

This cell displays and validates the fixed shared settings. The protocol is 50 epochs per condition (150 total), batch size 2, four gradient-accumulation steps, AdamW at 1e-4 with 1e-5 weight decay, AMP enabled, and the saved train/validation manifests. No scheduler, early stopping, gradient clipping, or silent memory-driven adjustment is introduced.

In [ ]:
from dataclasses import asdict
from neurolens.utils.reproducibility import seed_everything

EXPERIMENT_ROOT = DRIVE_PROJECT / "experiments"
CONDITIONS = ("A", "B", "C")
assert cfg.epochs == 50 and cfg.batch_size == 2 and cfg.accumulation_steps == 4
assert cfg.modalities == ("t1n", "t1c", "t2w", "t2f") and cfg.patch_size == (64, 64, 64)
assert cfg.learning_rate == 1e-4 and cfg.weight_decay == 1e-5 and cfg.base_channels == 8
assert cfg.seed == 42 and cfg.device == "cuda"
print("Shared experiment config:", json.dumps(asdict(cfg), indent=2, default=list))
print("Output root:", EXPERIMENT_ROOT)
print("GPU requirement: master execution refuses to begin unless CUDA is available.")


## 13. Master A/B/C training orchestrator

Run All invokes this cell. The single experiment control defaults to true. It checks CUDA before training, then runs conditions sequentially. Each condition gets a fresh seeded model, optimizer, GradScaler and DataLoader generator. Validation and test use deterministic slices. The test loader is not created until A, B and C have finished and their best-validation checkpoints are selected. On a rerun, atomic checkpoint state identifies each finished epoch and completed condition; the first incomplete condition resumes automatically.

In [ ]:
# This is the one master experiment control. Run All starts/resumes A, B, C.
RUN_ALL_CONDITIONS = True

import csv, datetime, hashlib, json, math, os, random, shutil, time, traceback
import numpy as np
import torch
from torch.utils.data import DataLoader
from neurolens.models.multitask import NeuroLens
from neurolens.training.trainer import Trainer
from neurolens.training.checkpoint import save_checkpoint, load_checkpoint
from neurolens.evaluation.segmentation import segmentation_metrics
from neurolens.evaluation.visualization import save_segmentation_overlay
from neurolens.utils.metrics import classification_metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_PATH = EXPERIMENT_ROOT / "experiment_status.json"
CONDITION_NAMES = {"A": "condition_A", "B": "condition_B", "C": "condition_C"}

def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray): return json_safe(value.tolist())
    if isinstance(value, np.generic): return json_safe(value.item())
    if isinstance(value, Path): return str(value)
    if isinstance(value, float) and not math.isfinite(value): return None
    return value

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(value), indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def write_history_csv(path, history):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if not history: return
    fields = list(history[0])
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader(); writer.writerows(history)
    temporary.replace(path)

def config_for_condition(condition):
    cfg.alignment_condition = condition
    cfg.output_dir = str(EXPERIMENT_ROOT / CONDITION_NAMES[condition])
    cfg.device = "cuda"
    cfg.validate()
    return json.loads(json.dumps(asdict(cfg), default=list))

def get_status():
    if STATUS_PATH.is_file():
        try: status = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
        except Exception as exc: raise RuntimeError(f"Cannot read status file {STATUS_PATH}: {exc}") from exc
    else:
        status = {"conditions": {}, "study_complete": False}
    if not isinstance(status.get("conditions"), dict):
        raise ValueError(f"Invalid experiment status schema in {STATUS_PATH}")
    for condition in CONDITIONS:
        status["conditions"].setdefault(condition, {"status": "pending", "last_completed_epoch": 0,
            "best_epoch": None, "validation_complete": False, "test_evaluated": False})
    status["study_complete"] = bool(status.get("study_complete", False) and
        all(status["conditions"][c].get("test_evaluated") for c in CONDITIONS))
    return status

status = get_status()
atomic_json(STATUS_PATH, status)

def set_status(condition, **updates):
    status["conditions"][condition].update(updates)
    status["updated_at_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    atomic_json(STATUS_PATH, status)

def expected_weights(condition):
    return {"classification": cfg.lambda_classification,
            "segmentation": cfg.lambda_segmentation,
            "contrastive": cfg.lambda_contrastive if condition in ("B", "C") else 0.0,
            "consistency": cfg.lambda_consistency if condition == "C" else 0.0}

def restore_random_states(extra):
    random.setstate(extra["python_rng"])
    np.random.set_state(extra["numpy_rng"])
    if extra.get("cuda_rng") is not None: torch.cuda.set_rng_state_all(extra["cuda_rng"])

def checkpoint_epoch(path):
    try:
        state = torch.load(path, map_location="cpu", weights_only=False)
        return int(state["epoch"]), state
    except Exception as exc:
        raise RuntimeError(f"Checkpoint is corrupt or unreadable: {path}: {type(exc).__name__}: {exc}") from exc

def evaluate_split(model, loader, device, condition, split_name, output_dir=None):
    """One deterministic pass; classification is global, segmentation is patient-macro."""
    model.eval()
    y_true, y_pred, probabilities, paired_cosines = [], [], [], []
    case_rows, per_class = [], {1: {"dice": [], "iou": []}, 2: {"dice": [], "iou": []}, 3: {"dice": [], "iou": []}}
    preview_written = False
    with torch.inference_mode():
        for batch_index, batch in enumerate(loader):
            try:
                x2 = batch["image2d"].to(device, non_blocking=True).float()
                x3 = batch["image3d"].to(device, non_blocking=True).float()
                output = model(x2, x3)
                probs = output["classification_logits"].softmax(1).cpu().numpy()
                predictions = probs.argmax(1)
                truth_class = batch["class_label"].cpu().numpy()
                y_true.extend(truth_class.tolist()); y_pred.extend(predictions.tolist()); probabilities.extend(probs.tolist())
                cosine = torch.nn.functional.cosine_similarity(output["embedding2d"], output["embedding3d"], dim=1)
                paired_cosines.extend(cosine.cpu().tolist())
                predicted_seg = output["segmentation_logits"].argmax(1).cpu().numpy()
                target_seg = batch["segmentation"].cpu().numpy()
                for i, patient_id in enumerate(batch["patient_id"]):
                    seg_metrics = segmentation_metrics(predicted_seg[i], target_seg[i])
                    class_metrics = {}
                    for label_id, label_name in ((1, "NCR"), (2, "ED"), (3, "ET")):
                        pm, tm = predicted_seg[i] == label_id, target_seg[i] == label_id
                        intersection = np.logical_and(pm, tm).sum(); denominator = pm.sum() + tm.sum()
                        union = np.logical_or(pm, tm).sum()
                        dice = float(2 * intersection / denominator) if denominator else 1.0
                        iou = float(intersection / union) if union else 1.0
                        per_class[label_id]["dice"].append(dice); per_class[label_id]["iou"].append(iou)
                        class_metrics[label_name] = {"dice": dice, "iou": iou}
                    case = {"patient_id": patient_id,
                            "classification_true": int(truth_class[i]),
                            "classification_pred": int(predictions[i]),
                            "classification_probabilities": probs[i].tolist(),
                            "segmentation": seg_metrics, "segmentation_per_class": class_metrics}
                    case_rows.append(case)
                    if output_dir is not None:
                        pred_dir = Path(output_dir) / "per_case"; pred_dir.mkdir(parents=True, exist_ok=True)
                        artifact = pred_dir / f"{patient_id}.npz"
                        tmp = artifact.with_suffix(".npz.tmp")
                        with tmp.open("wb") as stream:
                            np.savez_compressed(stream, patient_id=np.asarray(patient_id),
                                classification_true=truth_class[i], classification_pred=predictions[i],
                                classification_probabilities=probs[i],
                                segmentation_prediction=predicted_seg[i].astype(np.uint8),
                                segmentation_target=target_seg[i].astype(np.uint8),
                                input_t1n=x3[i, 0].cpu().numpy().astype(np.float32),
                                slice_index=int(batch["slice_index"][i]),
                                axis_order=np.asarray("(D,H,W)=(Z,Y,X), cropped patch space"))
                        tmp.replace(artifact)
                        if not preview_written:
                            index = int(batch["slice_index"][i])
                            slice_pred = np.take(predicted_seg[i], index, axis=cfg.slice_axis)
                            preview = Path(output_dir) / "qualitative" / f"{patient_id}_overlay.png"
                            preview.parent.mkdir(parents=True, exist_ok=True)
                            save_segmentation_overlay(batch["image2d"][i, 0].cpu().numpy(), slice_pred, preview, index)
                            preview_written = True
            except Exception as exc:
                patient = batch.get("patient_id", ["unknown"])[0]
                raise RuntimeError(f"{condition} {split_name} evaluation failed at batch {batch_index}, patient {patient}: {exc}") from exc
    cls = classification_metrics(y_true, y_pred, probabilities)
    class_summary = {name: {metric: float(np.mean(per_class[label][metric])) for metric in ("dice", "iou")}
                     for label, name in ((1, "NCR"), (2, "ED"), (3, "ET"))}
    seg_macro = {metric: float(np.nanmean([case["segmentation"][metric] for case in case_rows]))
                 for metric in ("dice", "iou", "hd95")}
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return json_safe({"condition": condition, "split": split_name, "patient_count": len(case_rows),
        "classification": cls, "classification_target": {"0": "no tumour", "1": "tumour present"},
        "confusion_matrix_labels": ["no tumour", "tumour"], "confusion_matrix": matrix,
        "segmentation_foreground_macro": seg_macro, "segmentation_per_class": class_summary,
        "segmentation_aggregation": "For each patient, foreground labels NCR/ED/ET are macro-averaged; then patient values are averaged.",
        "hd95_assumption": "Repository segmentation_metrics default spacing=(1,1,1), so HD95 is in voxel units; undefined empty-region values are NaN and serialized as null.",
        "alignment": {"paired_cosine_similarity": float(np.mean(paired_cosines)) if paired_cosines else None,
                      "n_pairs": len(paired_cosines), "meaning": "Descriptive for A; optimized by contrastive objective in B/C."},
        "cases": case_rows})

def save_confusion_figure(result, path):
    matrix = np.asarray(result["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(matrix, display_labels=result["confusion_matrix_labels"]).plot(ax=ax, colorbar=False)
    fig.tight_layout(); path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix); fig.savefig(tmp, dpi=150); plt.close(fig); tmp.replace(path)

if RUN_ALL_CONDITIONS:
    if not torch.cuda.is_available():
        raise RuntimeError("RUN_ALL_CONDITIONS=True requires a CUDA GPU. Select a Colab GPU runtime; this study will not fall back to CPU.")
    if split_counts != {"train": 296, "validation": 37, "test": 38}:
        raise RuntimeError(f"Saved BraTS split counts changed unexpectedly: {split_counts}")
    if train_dataset.random_slice is not True or val_dataset.random_slice is not False:
        raise RuntimeError("Dataset slice policy mismatch: training must be stochastic, validation deterministic.")
    if cfg.batch_size < 2: raise ValueError("Conditions B/C require at least two patients per batch for InfoNCE negatives.")
    val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                            pin_memory=True, drop_last=False)
    initial_parameter_digest = None
    for condition in CONDITIONS:
        condition_dir = EXPERIMENT_ROOT / CONDITION_NAMES[condition]
        evaluation_dir = condition_dir / "evaluation"
        condition_config = config_for_condition(condition)
        weights = expected_weights(condition)
        config_file = condition_dir / "config.json"
        condition_dir.mkdir(parents=True, exist_ok=True)
        if config_file.exists():
            existing_config = json.loads(config_file.read_text(encoding="utf-8"))
            if existing_config != condition_config:
                raise ValueError(f"{condition} config mismatch at {config_file}; refusing to resume with altered experimental settings.")
        else:
            atomic_json(config_file, condition_config)
            import yaml
            yaml_path = condition_dir / "runtime_config.yaml"
            temp_yaml = yaml_path.with_suffix(".yaml.tmp")
            temp_yaml.write_text(yaml.safe_dump(condition_config, sort_keys=False), encoding="utf-8")
            temp_yaml.replace(yaml_path)
        if not (condition_dir / "runtime_config.yaml").is_file():
            import yaml
            yaml_path = condition_dir / "runtime_config.yaml"; temp_yaml = yaml_path.with_suffix(".yaml.tmp")
            temp_yaml.write_text(yaml.safe_dump(condition_config, sort_keys=False), encoding="utf-8"); temp_yaml.replace(yaml_path)
        status["conditions"][condition].update({"condition": condition, "config": str(config_file)})
        state_path = condition_dir / "training_state.pt"
        latest_path = condition_dir / "latest.pt"
        best_path = condition_dir / "best_validation.pt"
        val_result_path = evaluation_dir / "validation_results.json"
        current = status["conditions"][condition]
        if current.get("validation_complete") and val_result_path.is_file() and best_path.is_file():
            if not (evaluation_dir / "validation_confusion_matrix.png").is_file():
                saved_validation = json.loads(val_result_path.read_text(encoding="utf-8"))
                save_confusion_figure(saved_validation, evaluation_dir / "validation_confusion_matrix.png")
            required_outputs = (state_path, latest_path, condition_dir / "training.csv", config_file,
                               condition_dir / "runtime_config.yaml", condition_dir / "run_metadata.json",
                               evaluation_dir / "validation_confusion_matrix.png")
            missing_outputs = [str(item) for item in required_outputs if not item.is_file()]
            if missing_outputs: raise FileNotFoundError(f"{condition} marked complete but required outputs are missing: {missing_outputs}")
            if int(current.get("last_completed_epoch", 0)) != cfg.epochs:
                raise RuntimeError(f"{condition} is marked validated but only {current.get('last_completed_epoch')} epochs are recorded; expected {cfg.epochs}.")
            state_epoch, state_payload = checkpoint_epoch(state_path)
            latest_epoch, latest_payload = checkpoint_epoch(latest_path)
            best_epoch, best_payload = checkpoint_epoch(best_path)
            if state_epoch != cfg.epochs or latest_epoch != state_epoch or state_payload.get("config") != condition_config or latest_payload.get("config") != condition_config:
                raise RuntimeError(f"{condition} latest/canonical checkpoint state is incompatible with the completed experiment status.")
            if best_epoch != int(current["best_epoch"]) or best_payload.get("config") != condition_config:
                raise RuntimeError(f"{condition} best checkpoint/config is inconsistent with experiment status.")
            state_extra = state_payload.get("extra_state")
            if not isinstance(state_extra, dict) or state_extra.get("epoch") != state_epoch or len(state_extra.get("history", [])) != state_epoch:
                raise RuntimeError(f"{condition} canonical recovery metadata is incomplete or corrupt.")
            saved_validation = json.loads(val_result_path.read_text(encoding="utf-8"))
            if saved_validation.get("condition") != condition or saved_validation.get("split") != "validation" or saved_validation.get("patient_count") != 37 or saved_validation.get("best_epoch") != best_epoch:
                raise RuntimeError(f"{condition} final validation report is inconsistent: {val_result_path}")
            digest = current.get("initial_parameter_sha256")
            if not digest: raise RuntimeError(f"{condition} status is missing its initialization digest.")
            if initial_parameter_digest is None: initial_parameter_digest = digest
            elif digest != initial_parameter_digest: raise RuntimeError(f"{condition} initialization digest differs across conditions.")
            print(f"{condition}: training and final validation already complete; skipping to next condition.")
            continue

        seed_everything(cfg.seed)
        cfg.alignment_condition = condition; cfg.output_dir = str(condition_dir); cfg.device = "cuda"
        model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
        digest = hashlib.sha256()
        for tensor in model.state_dict().values(): digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
        digest = digest.hexdigest()
        if initial_parameter_digest is None: initial_parameter_digest = digest
        elif digest != initial_parameter_digest: raise RuntimeError(f"{condition} initialization differs despite resetting seed {cfg.seed}.")
        trainer = Trainer(model, cfg)
        if trainer.loss.weights != weights:
            raise RuntimeError(f"{condition} Trainer loss weights mismatch: {trainer.loss.weights} != {weights}")
        loader_generator = torch.Generator(); loader_generator.manual_seed(cfg.seed)
        train_loader_condition = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
            num_workers=cfg.num_workers, pin_memory=True, drop_last=(condition in ("B", "C")), generator=loader_generator)
        if condition in ("B", "C") and (cfg.batch_size < 2 or not train_loader_condition.drop_last):
            raise RuntimeError(f"{condition} aligned training must use batch_size>=2 and drop_last=True.")
        if condition in ("B", "C") and len(train_loader_condition) != len(train_dataset) // cfg.batch_size:
            raise RuntimeError(f"{condition} aligned loader batch count is inconsistent with drop_last=True.")
        if len(train_dataset) != 296 or len(val_dataset) != 37:
            raise RuntimeError(f"{condition} dataset sizes differ from saved split: train={len(train_dataset)}, val={len(val_dataset)}")

        history, best_val, best_epoch, last_epoch, elapsed_before = [], float("inf"), None, 0, 0.0
        if state_path.exists():
            try:
                payload = torch.load(state_path, map_location="cpu", weights_only=False)
                if payload.get("config") != condition_config:
                    raise ValueError("checkpoint configuration is incompatible")
                last_epoch = load_checkpoint(state_path, trainer.model, trainer.optimizer, trainer.scaler, map_location="cpu")
                extra = payload.get("extra_state")
                if not isinstance(extra, dict): raise ValueError("checkpoint lacks recovery extra_state")
                if extra.get("condition") != condition or int(extra.get("epoch", -1)) != last_epoch:
                    raise ValueError("checkpoint condition/epoch metadata is inconsistent")
                history, best_val, best_epoch = extra["history"], float(extra["best_val"]), extra["best_epoch"]
                if extra.get("initial_parameter_sha256") != digest: raise ValueError("initial model state digest differs from resumed checkpoint config/seed")
                if len(history) != last_epoch or [int(row["epoch"]) for row in history] != list(range(1, last_epoch + 1)):
                    raise ValueError("checkpoint history does not contain exactly one row per completed epoch")
                elapsed_before = float(extra.get("elapsed_seconds", 0.0))
                restore_random_states(extra)
                if "loader_rng" not in extra: raise ValueError("checkpoint lacks DataLoader generator state")
                loader_generator.set_state(extra["loader_rng"])
                if last_epoch > cfg.epochs: raise ValueError(f"checkpoint epoch {last_epoch} exceeds configured epoch budget {cfg.epochs}")
                if latest_path.exists():
                    latest_epoch, latest_payload = checkpoint_epoch(latest_path)
                    if latest_payload.get("config") != condition_config: raise ValueError("latest.pt configuration is incompatible")
                    if latest_epoch > last_epoch: raise ValueError(f"latest.pt epoch {latest_epoch} is ahead of canonical state {last_epoch}")
                save_checkpoint(latest_path, trainer.model, trainer.optimizer, trainer.scaler, last_epoch, condition_config, extra)
                if best_epoch is not None:
                    if best_path.exists():
                        best_file_epoch, best_payload = checkpoint_epoch(best_path)
                        if best_payload.get("config") != condition_config: raise ValueError("best_validation.pt configuration is incompatible")
                        if best_file_epoch != int(best_epoch):
                            if int(best_epoch) != last_epoch: raise ValueError(f"best checkpoint epoch {best_file_epoch} disagrees with state {best_epoch}")
                            save_checkpoint(best_path, trainer.model, trainer.optimizer, trainer.scaler, last_epoch, condition_config, extra)
                    elif int(best_epoch) == last_epoch:
                        save_checkpoint(best_path, trainer.model, trainer.optimizer, trainer.scaler, last_epoch, condition_config, extra)
                    else:
                        raise FileNotFoundError(f"Best checkpoint {best_path} is missing and cannot be reconstructed from the latest model.")
                elif best_path.exists():
                    raise ValueError("best_validation.pt exists but recovery state has no best epoch")
                print(f"{condition}: resumed from epoch {last_epoch}; best validation loss={best_val}; best epoch={best_epoch}.")
            except Exception as exc:
                raise RuntimeError(f"{condition} resume failed from {state_path}: {type(exc).__name__}: {exc}. Preserve files and inspect checkpoint/config compatibility.") from exc
        elif latest_path.exists() or best_path.exists():
            raise RuntimeError(f"{condition} has model checkpoints but no canonical recovery state {state_path}; refusing to restart from scratch.")
        write_history_csv(condition_dir / "training.csv", history)
        started = time.time()
        set_status(condition, status="running", last_completed_epoch=last_epoch, best_epoch=best_epoch,
                   validation_complete=False, error=None)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        for epoch in range(last_epoch + 1, cfg.epochs + 1):
            try:
                train_metrics = trainer.run_epoch(train_loader_condition, training=True)
                # Exactly one deterministic validation forward pass per epoch returns
                # both the loss terms and globally/per-patient aggregated metrics.
                val_metrics = trainer.run_epoch(val_loader, training=False, collect_metrics=True)
                row = {"epoch": epoch, **{f"train_{k}": float(v) for k, v in train_metrics.items()},
                       **{f"val_{k}": float(v) for k, v in val_metrics.items()}}
                invalid = {k: v for k, v in row.items() if not math.isfinite(v)}
                if invalid: raise FloatingPointError(f"non-finite losses/metrics: {invalid}")
            except Exception as exc:
                set_status(condition, status="failed", last_completed_epoch=epoch - 1,
                           error=f"epoch {epoch}: {type(exc).__name__}: {exc}")
                raise RuntimeError(f"{condition} epoch {epoch} failed; prior recovery checkpoint remains at {state_path}: {type(exc).__name__}: {exc}") from exc
            history.append(row)
            improved = val_metrics["total"] < best_val
            if improved: best_val, best_epoch = val_metrics["total"], epoch
            elapsed_total = elapsed_before + (time.time() - started)
            extra = {"condition": condition, "epoch": epoch, "history": history,
                "best_val": best_val, "best_epoch": best_epoch, "elapsed_seconds": elapsed_total,
                "initial_parameter_sha256": digest,
                "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                "loader_rng": loader_generator.get_state()}
            # training_state.pt is the canonical atomic recovery record. Other files
            # are mirrors and are repaired from it if Colab disconnects mid-write.
            save_checkpoint(state_path, trainer.model, trainer.optimizer, trainer.scaler, epoch, condition_config, extra)
            save_checkpoint(latest_path, trainer.model, trainer.optimizer, trainer.scaler, epoch, condition_config, extra)
            if improved:
                save_checkpoint(best_path, trainer.model, trainer.optimizer, trainer.scaler, epoch, condition_config, extra)
            write_history_csv(condition_dir / "training.csv", history)
            peak_vram = torch.cuda.max_memory_allocated() / 2**30
            set_status(condition, status="running", last_completed_epoch=epoch, best_epoch=best_epoch,
                       best_validation_loss=best_val, validation_complete=False, peak_gpu_memory_gib=peak_vram,
                       initial_parameter_sha256=digest)
            atomic_json(condition_dir / "run_metadata.json", {"condition": condition, "seed": cfg.seed,
                "device": torch.cuda.get_device_name(0), "epochs_completed": epoch,
                "initial_parameter_sha256": digest,
                "elapsed_seconds": elapsed_total, "peak_gpu_memory_gib": peak_vram,
                "optimizer": "AdamW", "learning_rate": cfg.learning_rate, "weight_decay": cfg.weight_decay,
                "scheduler": None, "early_stopping": None, "gradient_clipping": None})
            print(f"{condition} epoch {epoch}/{cfg.epochs}: train_total={train_metrics['total']:.5f} "
                  f"val_total={val_metrics['total']:.5f} val_acc={val_metrics['classification_accuracy']:.4f} "
                  f"val_f1={val_metrics['classification_f1']:.4f} val_dice={val_metrics['segmentation_dice']:.4f} "
                  f"best_epoch={best_epoch} peak_vram={peak_vram:.2f} GiB")
        # A resumed, already-50-epoch state skips the loop and proceeds to final validation.
        if last_epoch >= cfg.epochs:
            print(f"{condition}: all {cfg.epochs} epochs already checkpointed; final validation will be verified.")
        metadata_path = condition_dir / "run_metadata.json"
        previous_metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
        elapsed_total = max(float(previous_metadata.get("elapsed_seconds", 0.0)), elapsed_before + (time.time() - started))
        peak_vram = max(float(previous_metadata.get("peak_gpu_memory_gib", 0.0)), torch.cuda.max_memory_allocated() / 2**30)
        atomic_json(metadata_path, {"condition": condition, "seed": cfg.seed,
            "device": torch.cuda.get_device_name(0), "epochs_completed": last_epoch if last_epoch >= cfg.epochs else cfg.epochs,
            "elapsed_seconds": elapsed_total, "peak_gpu_memory_gib": peak_vram,
            "initial_parameter_sha256": digest, "optimizer": "AdamW", "learning_rate": cfg.learning_rate,
            "weight_decay": cfg.weight_decay, "scheduler": None, "early_stopping": None,
            "gradient_clipping": None, "amp": cfg.mixed_precision})
        if not best_path.is_file(): raise FileNotFoundError(f"{condition} has no best-validation checkpoint: {best_path}")
        if val_result_path.is_file():
            try:
                existing_val = json.loads(val_result_path.read_text(encoding="utf-8"))
                existing_valid = (existing_val.get("condition") == condition and existing_val.get("split") == "validation"
                                  and existing_val.get("patient_count") == 37 and existing_val.get("best_epoch") == int(best_epoch))
            except Exception as exc:
                raise RuntimeError(f"{condition} validation report is corrupt at {val_result_path}: {exc}") from exc
            if existing_valid:
                save_confusion_figure(existing_val, evaluation_dir / "validation_confusion_matrix.png")
                set_status(condition, status="trained_validated", last_completed_epoch=cfg.epochs,
                           best_epoch=int(best_epoch), validation_complete=True, test_evaluated=False,
                           initial_parameter_sha256=digest, error=None)
                del model, trainer, train_loader_condition
                torch.cuda.empty_cache()
                print(f"{condition}: recovered the already-written final validation report; skipping repeated validation decoding.")
                continue
        best_model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
        best_epoch_loaded = load_checkpoint(best_path, best_model, map_location="cpu")
        if best_epoch_loaded != int(best_epoch): raise RuntimeError(f"{condition} best checkpoint epoch mismatch: {best_epoch_loaded} != {best_epoch}")
        best_model.to("cuda").eval()
        val_result = evaluate_split(best_model, val_loader, torch.device("cuda"), condition, "validation")
        if {row["patient_id"] for row in val_result["cases"]} != split_ids["validation"]:
            raise RuntimeError(f"{condition} validation evaluation did not cover the exact saved 37-patient validation set.")
        val_result.update({"best_validation_loss": float(best_val), "best_epoch": int(best_epoch),
                           "checkpoint": str(best_path), "metrics_from_best_checkpoint": True})
        atomic_json(val_result_path, val_result)
        save_confusion_figure(val_result, evaluation_dir / "validation_confusion_matrix.png")
        set_status(condition, status="trained_validated", last_completed_epoch=cfg.epochs,
                   best_epoch=int(best_epoch), validation_complete=True, test_evaluated=False,
                   initial_parameter_sha256=digest, error=None)
        del best_model, model, trainer, train_loader_condition
        torch.cuda.empty_cache()

    # Test access is gated on all three best checkpoints and final validation reports.
    for condition in CONDITIONS:
        cstate = status["conditions"][condition]
        if not cstate.get("validation_complete") or not (EXPERIMENT_ROOT / CONDITION_NAMES[condition] / "best_validation.pt").is_file():
            raise RuntimeError(f"Held-out test evaluation blocked: {condition} has not completed training and best-checkpoint validation.")
    test_dataset = make_test_dataset(cfg)
    if test_dataset.random_slice is not False or len(test_dataset) != 38:
        raise RuntimeError(f"Test dataset must be the deterministic saved 38-patient split; got n={len(test_dataset)}, random_slice={test_dataset.random_slice}")
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                             pin_memory=True, drop_last=False)
    test_ids = set(read_patient_ids(cfg.manifest_test, cfg.manifest_id_column))
    if test_ids != split_ids["test"]: raise RuntimeError("Test manifest patient IDs changed during experiment execution.")
    for condition in CONDITIONS:
        condition_dir = EXPERIMENT_ROOT / CONDITION_NAMES[condition]
        result_path = condition_dir / "evaluation" / "test_results.json"
        if status["conditions"][condition].get("test_evaluated") and not result_path.is_file():
            raise FileNotFoundError(f"{condition} status marks test evaluation complete but report is missing: {result_path}")
        if result_path.is_file():
            try:
                prior_test = json.loads(result_path.read_text(encoding="utf-8"))
                selected_checkpoint_epoch, selected_payload = checkpoint_epoch(condition_dir / "best_validation.pt")
                prior_ids = {row.get("patient_id") for row in prior_test.get("cases", [])}
                artifacts_complete = all((condition_dir / "evaluation" / "per_case" / f"{patient_id}.npz").is_file() for patient_id in test_ids)
                complete_prior = (prior_test.get("condition") == condition and prior_test.get("split") == "test"
                                  and prior_test.get("patient_count") == 38 and prior_test.get("selected_epoch") == selected_checkpoint_epoch
                                  and prior_ids == test_ids and artifacts_complete
                                  and selected_payload.get("config") == config_for_condition(condition))
            except Exception as exc:
                raise RuntimeError(f"{condition} existing test report is corrupt or incompatible at {result_path}: {exc}") from exc
            if complete_prior:
                save_confusion_figure(prior_test, condition_dir / "evaluation" / "test_confusion_matrix.png")
                qualitative_dir = condition_dir / "evaluation" / "qualitative"; qualitative_dir.mkdir(parents=True, exist_ok=True)
                for artifact_path in sorted((condition_dir / "evaluation" / "per_case").glob("*.npz"))[:1]:
                    with np.load(artifact_path) as artifact:
                        idx = int(artifact["slice_index"])
                        view = np.take(artifact["input_t1n"], idx, axis=cfg.slice_axis)
                        pred_view = np.take(artifact["segmentation_prediction"], idx, axis=cfg.slice_axis)
                        save_segmentation_overlay(view, pred_view, qualitative_dir / f"{artifact['patient_id'].item()}_overlay.png", idx)
                set_status(condition, test_evaluated=True, test_patient_count=38)
                print(f"{condition}: complete final test evaluation already recorded; skipping."); continue
            if status["conditions"][condition].get("test_evaluated"):
                raise RuntimeError(f"{condition} status says test evaluation completed, but {result_path} is incomplete/inconsistent.")
            raise RuntimeError(f"{condition} has an incomplete existing test report at {result_path}; preserve it and inspect before rerunning.")
        model = NeuroLens(len(cfg.modalities), cfg.base_channels, cfg.projection_dim, cfg.num_classes, cfg.classification_classes)
        best_path = condition_dir / "best_validation.pt"
        selected_epoch = load_checkpoint(best_path, model, map_location="cpu")
        model.to("cuda").eval()
        result = evaluate_split(model, test_loader, torch.device("cuda"), condition, "test", condition_dir / "evaluation")
        if {row["patient_id"] for row in result["cases"]} != test_ids:
            raise RuntimeError(f"{condition} test evaluation did not cover the exact saved 38-patient test set.")
        result.update({"selected_epoch": selected_epoch, "checkpoint": str(best_path),
                       "test_usage": "Held-out final evaluation only; never used for selection."})
        atomic_json(result_path, result)
        save_confusion_figure(result, condition_dir / "evaluation" / "test_confusion_matrix.png")
        set_status(condition, test_evaluated=True, test_patient_count=result["patient_count"])
        del model; torch.cuda.empty_cache()
        print(condition, "test evaluation saved:", result_path)

    if all(status["conditions"][c].get("test_evaluated") for c in CONDITIONS):
        print("A, B, and C training, final validation, and held-out test evaluations are complete.")
else:
    print("RUN_ALL_CONDITIONS=False; no experiment was started.")


## 14. Validation, checkpoint selection, and automatic recovery

Each epoch delegates training and validation loss computation to `Trainer.run_epoch`. Validation uses `training=False`, which sets `model.eval()` and disables gradients. Its optional metric collection reuses the same forward pass, so there is no second validation volume decode per epoch. Classification precision/recall/F1 are weighted by class support and computed globally from all patient predictions. Segmentation labels 1–3 are macro-averaged within each patient and then averaged over patients; empty-empty Dice/IoU is 1, matching the existing utility. Epoch HD95 is omitted for performance; final validation/test use the repository `segmentation_metrics` implementation (default unit voxel spacing; undefined values become JSON null).

The best checkpoint is selected only by lowest complete validation total loss. `training_state.pt` is the canonical atomic recovery checkpoint with model, AdamW, GradScaler, epoch, config, all history, best state, elapsed time, and Python/NumPy/Torch/CUDA RNG states. `latest.pt` and `best_validation.pt` are atomically written mirrors. A missing latest/best mirror can be repaired from canonical state when the epoch permits; corrupt, incompatible, or irrecoverably missing best state stops with an explicit error. CSV is rebuilt atomically from checkpoint history to prevent duplicate epoch rows. A completed condition is skipped on rerun.

## 15. Final held-out test evaluation

The master cell waits until all three conditions have completed 50 epochs and each best-validation checkpoint has been reloaded and evaluated on all 37 validation patients. Only then does it instantiate the saved 38-patient test dataset. The same deterministic dataset and ordered DataLoader are used for A, B, and C. Each test result and per-case prediction is saved under that condition. Restarting after interruption skips conditions with an atomically written test report; no test metric changes checkpoint selection.

## 16. Comparative metrics, plots, and descriptive analysis

When all three test reports exist, this section creates the requested comparison table, CSV/JSON, training and validation loss curves, validation classification F1 and segmentation Dice curves, per-class Dice and alignment plots, and confusion-matrix copies. It also calculates paired per-patient mean differences with bootstrap confidence intervals as descriptive summaries only; no statistical significance claim is made. Missing/non-finite metrics remain null and are not replaced with zero.

In [ ]:
import csv, json, math, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

COMPARISON_ROOT = EXPERIMENT_ROOT / "comparative_analysis"
PLOT_ROOT = COMPARISON_ROOT / "comparison_plots"
CM_ROOT = COMPARISON_ROOT / "confusion_matrices"
QUAL_ROOT = COMPARISON_ROOT / "qualitative_results"
for folder in (PLOT_ROOT, CM_ROOT, QUAL_ROOT): folder.mkdir(parents=True, exist_ok=True)
if not all((EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation" / "validation_results.json").is_file()
           and (EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation" / "test_results.json").is_file() for c in CONDITIONS):
    print("Comparative analysis is pending: at least one condition's final validation/test report is missing.")
else:
    reports = {c: {
        "validation": json.loads((EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation/validation_results.json").read_text()),
        "test": json.loads((EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation/test_results.json").read_text()),
        "history": list(csv.DictReader((EXPERIMENT_ROOT / CONDITION_NAMES[c] / "training.csv").open(newline="", encoding="utf-8")))
    } for c in CONDITIONS}
    def get_metric(report, split, *keys):
        value = report[split]
        for key in keys: value = value.get(key) if isinstance(value, dict) else None
        return value
    table = []
    for metric_name, getter in [
        ("Best validation loss", lambda c: reports[c]["validation"].get("best_validation_loss")),
        ("Validation classification accuracy", lambda c: get_metric(reports[c], "validation", "classification", "accuracy")),
        ("Validation classification precision", lambda c: get_metric(reports[c], "validation", "classification", "precision")),
        ("Validation classification recall", lambda c: get_metric(reports[c], "validation", "classification", "recall")),
        ("Validation classification F1", lambda c: get_metric(reports[c], "validation", "classification", "f1")),
        ("Validation foreground Dice", lambda c: get_metric(reports[c], "validation", "segmentation_foreground_macro", "dice")),
        ("Validation foreground IoU", lambda c: get_metric(reports[c], "validation", "segmentation_foreground_macro", "iou")),
        ("Validation NCR Dice", lambda c: get_metric(reports[c], "validation", "segmentation_per_class", "NCR", "dice")),
        ("Validation ED Dice", lambda c: get_metric(reports[c], "validation", "segmentation_per_class", "ED", "dice")),
        ("Validation ET Dice", lambda c: get_metric(reports[c], "validation", "segmentation_per_class", "ET", "dice")),
        ("Validation paired cosine", lambda c: get_metric(reports[c], "validation", "alignment", "paired_cosine_similarity")),
        ("Test classification F1", lambda c: get_metric(reports[c], "test", "classification", "f1")),
        ("Test foreground Dice", lambda c: get_metric(reports[c], "test", "segmentation_foreground_macro", "dice")),
        ("Test foreground IoU", lambda c: get_metric(reports[c], "test", "segmentation_foreground_macro", "iou")),
    ]:
        row = {"metric": metric_name}
        for c in CONDITIONS: row[c] = getter(c)
        table.append(row)
    table_path = COMPARISON_ROOT / "comparison.csv"
    table_tmp = table_path.with_suffix(".csv.tmp")
    with table_tmp.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["metric", *CONDITIONS]); writer.writeheader(); writer.writerows(table)
    table_tmp.replace(table_path)
    atomic_json(COMPARISON_ROOT / "comparison.json", {"conditions": reports, "table": table,
        "interpretation": "Descriptive comparison only. No significance claim is made."})

    def plot_metric(title, ylabel, values, filename):
        fig, ax = plt.subplots(figsize=(8, 5))
        for c in CONDITIONS:
            series = values[c]
            if series: ax.plot(range(1, len(series) + 1), series, label=f"Condition {c}")
        ax.set(title=title, xlabel="Epoch", ylabel=ylabel); ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
        fig.savefig(PLOT_ROOT / filename, dpi=150); plt.close(fig)
    for prefix, key, title in (("train", "total", "Training total loss"), ("val", "total", "Validation total loss"),
                               ("val", "classification_f1", "Validation classification F1"),
                               ("val", "segmentation_dice", "Validation foreground Dice")):
        values = {c: [float(row[f"{prefix}_{key}"]) for row in reports[c]["history"] if row.get(f"{prefix}_{key}") not in (None, "")]
                  for c in CONDITIONS}
        plot_metric(title, key.replace("_", " "), values, f"{prefix}_{key}.png")

    for split in ("validation", "test"):
        for c in CONDITIONS:
            source = EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation" / f"{split}_confusion_matrix.png"
            if source.is_file(): shutil.copy2(source, CM_ROOT / f"condition_{c}_{split}.png")
    for metric in ("dice", "iou"):
        names = ("NCR", "ED", "ET")
        x = np.arange(len(names)); width = .24
        fig, ax = plt.subplots(figsize=(8, 5))
        for j, c in enumerate(CONDITIONS):
            vals = [reports[c]["test"]["segmentation_per_class"][name][metric] for name in names]
            ax.bar(x + (j - 1) * width, vals, width, label=f"Condition {c}")
        ax.set_xticks(x, names); ax.set_ylabel(metric.upper()); ax.set_title(f"Held-out test per-class {metric.upper()}"); ax.legend(); fig.tight_layout()
        fig.savefig(PLOT_ROOT / f"test_per_class_{metric}.png", dpi=150); plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 4))
    for c in CONDITIONS:
        value = reports[c]["test"]["alignment"]["paired_cosine_similarity"]
        if value is not None: ax.bar(c, value, label=f"Condition {c}")
    ax.set(title="Held-out paired embedding cosine (descriptive)", ylabel="Cosine similarity"); fig.tight_layout()
    fig.savefig(PLOT_ROOT / "test_alignment_cosine.png", dpi=150); plt.close(fig)

    # Pair test cases by patient and bootstrap paired mean differences. These
    # intervals are descriptive and are not a significance test.
    rng = np.random.default_rng(cfg.seed); paired = {}
    case_maps = {c: {row["patient_id"]: row for row in reports[c]["test"]["cases"]} for c in CONDITIONS}
    if any(set(case_maps[c]) != split_ids["test"] for c in CONDITIONS): raise RuntimeError("Comparative cases do not exactly match the saved test patient set.")
    for c in CONDITIONS:
        for overlay in sorted((EXPERIMENT_ROOT / CONDITION_NAMES[c] / "evaluation" / "qualitative").glob("*_overlay.png")):
            shutil.copy2(overlay, QUAL_ROOT / f"condition_{c}_{overlay.name}")
    for left, right in (("B", "A"), ("C", "A"), ("C", "B")):
        common = sorted(set(case_maps[left]) & set(case_maps[right]))
        stats = {}
        metrics = {"classification_correct": lambda row: float(row["classification_true"] == row["classification_pred"]),
                   "segmentation_dice": lambda row: row["segmentation"]["dice"],
                   "segmentation_iou": lambda row: row["segmentation"]["iou"]}
        for name, get_value in metrics.items():
            differences = np.asarray([get_value(case_maps[left][pid]) - get_value(case_maps[right][pid]) for pid in common], dtype=float)
            draws = np.asarray([rng.choice(differences, size=len(differences), replace=True).mean() for _ in range(2000)])
            stats[name] = {"n_patients": len(common), "mean_difference": float(differences.mean()),
                           "descriptive_bootstrap_95pct_interval": [float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))]}
        paired[f"{left}-{right}"] = stats
    atomic_json(COMPARISON_ROOT / "paired_patient_differences.json", paired)
    print("Comparative table saved:", table_path)
    for row in table: print(row)


## 17. Qualitative segmentation analysis

The A/B/C evaluation pass writes one representative same-plane overlay per condition and compressed per-patient predictions/targets for the full test split. Browse `comparative_analysis/qualitative_results` and each condition’s `evaluation/qualitative` directory. Compare identical patient IDs and the same deterministic slice. Interpret visual errors alongside class-wise Dice/IoU; an overlay is illustrative, not a performance estimate.

In [ ]:
from IPython.display import display, Image as DisplayImage
qual_files = []
for condition in CONDITIONS:
    folder = EXPERIMENT_ROOT / CONDITION_NAMES[condition] / "evaluation" / "qualitative"
    qual_files.extend((condition, p) for p in sorted(folder.glob("*_overlay.png")))
if not qual_files:
    print("Qualitative overlays are generated after all three held-out evaluations complete.")
else:
    for condition, image_path in qual_files:
        print("Condition", condition, "—", image_path.name)
        display(DisplayImage(filename=str(image_path)))


## 18. Final research report and conclusions

The final report is written only when A/B/C have complete validation and held-out test results. It records configuration, dataset/split provenance, selected epochs, measured metrics, run times, checkpoint paths, comparative tables/plots, and per-patient descriptive differences. The experiment status JSON is updated after every committed epoch and evaluation stage. After an interruption, rerun the notebook from the top; completed conditions and reports are detected and skipped.

Interpretation template:

- **Measured baseline:** state A validation/test metrics from generated JSON.
- **Contrastive effect:** compare B with A on same patients; report descriptive paired intervals without a significance claim.
- **Consistency effect:** compare C with B.
- **Classification:** binary tumour presence, not tumour subtype.
- **Segmentation:** foreground macro plus NCR/ED/ET class-wise metrics.
- **Alignment:** paired cosine is descriptive for A and trained by alignment objectives for B/C.
- **Limitations:** cohort size, crop-space predictions, stochastic training-slice choice, unit-spacing HD95, compute interruptions.
- **Future work:** clearly label hypotheses and planned experiments separately from findings.

In [ ]:
from pathlib import Path
import csv, json

def final_report_text():
    lines = ["# NeuroLens Three-Condition Research Report", "",
        "## Experimental protocol", "- Architecture: repository NeuroLens 2D/3D dual-path model, unchanged.",
        "- Dataset: BraTS 2023 GLI; saved patient manifests; train 296, validation 37, test 38.",
        "- BRISC: audited independently; not used to create BraTS pairs or training batches.",
        "- Condition A/B/C: classification+segmentation; then +contrastive; then +feature consistency.",
        "- Selection: lowest complete validation total loss; test split used only for final evaluation.", ""]
    lines.extend(["## Condition status", "| Condition | Status | Epoch | Best epoch | Validation | Test |", "|---|---|---:|---:|---|---|"])
    status_now = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    for c in CONDITIONS:
        state = status_now["conditions"][c]
        lines.append(f"| {c} | {state.get('status')} | {state.get('last_completed_epoch')} | {state.get('best_epoch')} | {state.get('validation_complete')} | {state.get('test_evaluated')} |")
    lines.extend(["", "## Runtime and checkpoints"])
    for c in CONDITIONS:
        cdir = EXPERIMENT_ROOT / CONDITION_NAMES[c]
        metadata_path = cdir / "run_metadata.json"
        if metadata_path.is_file():
            meta = json.loads(metadata_path.read_text(encoding="utf-8"))
            lines.append(f"- **Condition {c}:** {meta.get('epochs_completed')} epochs; elapsed {meta.get('elapsed_seconds')} seconds; peak GPU memory {meta.get('peak_gpu_memory_gib')} GiB; device {meta.get('device')}.")
        else:
            lines.append(f"- **Condition {c}:** runtime metadata unavailable.")
        lines.append(f"  - Best checkpoint: `{cdir / 'best_validation.pt'}`; latest: `{cdir / 'latest.pt'}`; config: `{cdir / 'config.json'}`.")
    lines.append(f"- Training curves: `{EXPERIMENT_ROOT / 'comparative_analysis/comparison_plots'}`.")
    comparison_file = EXPERIMENT_ROOT / "comparative_analysis/comparison.csv"
    if comparison_file.is_file():
        lines.extend(["", "## Comparative results", "| Metric | A | B | C |", "|---|---:|---:|---:|"])
        with comparison_file.open(newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                vals = [row.get(c, "") if row.get(c) not in (None, "None", "nan") else "NA" for c in CONDITIONS]
                lines.append(f"| {row['metric']} | {vals[0]} | {vals[1]} | {vals[2]} |")
    else:
        lines.extend(["", "## Comparative results", "Not available until all condition evaluations are complete."])
    lines.extend(["", "## Outputs", f"- Experiment status: `{STATUS_PATH}`",
        f"- Conditions: `{EXPERIMENT_ROOT}`", f"- Comparative analysis: `{EXPERIMENT_ROOT / 'comparative_analysis'}`",
        "", "## Interpretation", "All reported differences are descriptive. No statistical significance is claimed.",
        "Classification is tumour presence. HD95 uses unit voxel spacing; null indicates an undefined metric.",
        "Only actual completed runs are reported; incomplete or missing outputs are not imputed."])
    return "\n".join(lines) + "\n"

REPORT_PATH = EXPERIMENT_ROOT / "comparative_analysis/final_research_report.md"
if RUN_ALL_CONDITIONS and STATUS_PATH.is_file():
    current_status = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    complete = all(current_status["conditions"][c].get("test_evaluated") for c in CONDITIONS)
    if complete:
        REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = REPORT_PATH.with_suffix(".md.tmp"); tmp.write_text(final_report_text(), encoding="utf-8"); tmp.replace(REPORT_PATH)
        current_status["study_complete"] = True; current_status["report"] = str(REPORT_PATH)
        atomic_json(STATUS_PATH, current_status)
        print("Study complete. Final report:", REPORT_PATH)
    else:
        print("Report not marked complete; one or more A/B/C test evaluations remain.")
